In [1]:
print("hello")

hello


In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import format_instructions
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

API key validation

In [3]:
from google.genai.errors import APIError
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
try:
    response = llm_gemini.invoke("Hello")
    print(response.content)
except Exception as e:
    print("Raw Error:", e)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPVLvwkU8I38G1tlB2hW1xzyAT/VfiiV8LWM1PjiT4ppJETTQcSGs9IXgDLbnPG3NvWG6AJOg51gs2CNOTmtk0qQnIzAknrx0fgC6swgjs9DF3jx2qb7pP'}}]


Envirnoment Variable Checking

In [4]:
if os.environ.get("GOOGLE_API_KEY"):
    print("api key is found that is gemini")
else:
    raise ValueError("GOOGLE_API_KEY environment variable not set")

api key is found that is gemini


Single Chain

In [ ]:
from langchain_core.output_parsers import format_instructions
# task 1
prompt=ChatPromptTemplate.from_messages([
    ("system","you are a teacher"),
    ("user"," Teach me about {topic}")
])

# task 2
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

# task 3
parser=StrOutputParser()

# Chain Creation
chain=prompt|llm_gemini|parser

# chain invocation
result=chain.invoke({"topic": "python"})
result


'Hello! Welcome to your very first Python lesson. I\'m so excited to be your teacher today. \n\nPython is one of the most popular programming languages in the world. It’s used for everything from building websites and analyzing data to creating artificial intelligence (like ChatGPT!). The best part? It is designed to be easy to read and write, almost like plain English.\n\nThink of learning Python like learning a new spoken language, but instead of talking to people, you are giving instructions to a computer.\n\nReady? Let’s begin with Lesson 1!\n\n---\n\n### Lesson 1: Your First "Conversation" with Python\n\nWhen humans want to say hello, we speak. When we want a computer to "speak," we use a command called `print()`. \n\nDon\'t let the word confuse you—it doesn\'t mean printing on a piece of paper. In programming, `print` means **"show this on the screen."**\n\nIf you want the computer to say "Hello, World!", you write it like this:\n\n```python\nprint("Hello, World!")\n```\n\n**Try 

Chain with Custom Function

In [15]:
# custum function
def custom_function(text:str)->str:
     return f"the answer is {text}"

user_input=input("give me a topic to teach")

#chain creation
custom_chain=prompt|llm_gemini|parser|custom_function

#chain invocation
custom_result=custom_chain.invoke({"topic": user_input})
custom_result

'the answer is Welcome to class! Take a seat. Today, we are going to talk about one of the most exciting and fast-moving topics in the world: **Artificial Intelligence**, or **AI**. \n\nDon\'t worry if it sounds complicated. By the end of this lesson, you’ll have a solid grasp of what it is, how it works, and why it\'s changing the world around us.\n\n---\n\n### Lesson 1: What *Is* AI, Exactly?\n\nIn simple terms, **Artificial Intelligence is the science of making computers and machines do things that normally require human intelligence.**\n\nThink about what humans can do: we learn from our mistakes, we recognize faces, we understand speech, we make decisions, and we create art. AI is an attempt to give computers those same abilities.\n\n**A quick pop-quiz thought:** Is a basic calculator AI? \n*Answer:* **No.** A calculator just follows strict, pre-written math rules very fast. It doesn\'t "learn" or "think." True AI can adapt, learn from data, and come up with answers it wasn’t expl

parallel chains

In [ ]:
#chain one
# task 1
prompt_One=ChatPromptTemplate.from_messages([
    ("system","you are a teacher"),
    ("user"," Teach me about {topic_one}")
    # {format_instructions}
])

# task 2
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

# task 3
strparser=StrOutputParser()

# task 4 {custum function}
def custom_function(text:str)->str:
     return f"the answer is \n {text}"

#chain two
#task 3
jsonparser=JsonOutputParser()

#task 1
prompt_two = ChatPromptTemplate.from_messages([
    ("system","you are a accurate ai assistent "),
    ("user"," tell me about {topic_two} {format_instructions}"),
    
]).partial(format_instructions=jsonparser.get_format_instructions())

#task 2
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

#chain creation

chain_one=prompt_One | llm_gemini | strparser | custom_function
chain_two = prompt_two | llm_gemini | jsonparser 
 
#chain invacation

parallel_chain = RunnableParallel(
    result1=chain_one,
    result2=chain_two
)
user_input_one=input("enter a topic to teach")
user_input_two=input("enter a topic to know about it")

parallel_chain.invoke({"topic_one":user_input_one,"topic_two":user_input_two})

{'result1': 'the answer is \n Hello! I love this question. Relationships are a very important part of growing up and understanding human connection. \n\nThink of having a girlfriend not as a status symbol, but as having a **close, special friendship** combined with romantic feelings, mutual respect, and care. \n\nLet’s break it down into four main parts: **What it is, How it starts, How to be a good partner,** and **Knowing when you\'re ready.**\n\n---\n\n### 1. What *is* a Girlfriend?\nA girlfriend is someone you choose to spend romantic time with because you enjoy each other’s company, trust one another, and care deeply for each other\'s happiness. It means going from "just friends" to a deeper level of commitment. \n\n### 2. How Do You Get One? (The Foundation)\nYou don\'t "win" a girlfriend like a prize; rather, you build a connection. It usually happens like this:\n* **Friendship First:** You get to know someone and realize you share common interests, laugh at the same things, and